# 基础功能测试：指定股票监控 + 实时新闻推送

这个 notebook 只测你现在最关心的两件事：

1. 指定股票是否会被监控，非监控股票是否会被忽略；
2. 来一条实时新闻后，是否会入库、评分，并触发推送。

默认测试不连接 IB、不真实推 Bark，而是用模拟 `tickNews` 和记录型 Bark 客户端做稳定测试。最后两节提供真实 Bark 和真实 IB 订阅开关。


## 0. 基础参数


In [9]:
from pathlib import Path
import os
import sqlite3
import sys
import tempfile
import time

cwd = Path.cwd()
package_dir = cwd if cwd.name == "news_api" else cwd / "news_api"
sys.path.insert(0, str(package_dir.parent))

TEST_SYMBOL = "ORCL"
TEST_WATCHLIST = {
    "ORCL": {
        "exchange": "NYSE",
        "priority": 0,
        "aliases": ["Oracle", "Oracle Corp", "OCI"],
    }
}

TEST_DB = Path(tempfile.mkdtemp()) / "basic_news_test.sqlite"
print("package_dir =", package_dir)
print("TEST_DB =", TEST_DB)


package_dir = e:\策略\IB-API\news_api
TEST_DB = C:\Users\Beyond\AppData\Local\Temp\tmpggzcu92m\basic_news_test.sqlite


## 1. 测试工具：记录型 Bark

这个类不走网络，只记录“本来应该推送什么”。如果这个测试通过，说明模块内部已经触发推送动作。真实 Bark 在后面单独测。


In [10]:
class RecordingBarkClient:
    def __init__(self):
        self.calls = []

    def push(self, analysis, priority):
        self.calls.append({
            "symbol": analysis.symbol,
            "headline": analysis.headline,
            "event_type": analysis.event_type,
            "importance_score": analysis.importance_score,
            "priority": priority,
        })
        return "ok", "recorded"


def read_table(db_path, table):
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(f"SELECT * FROM {table}").fetchall()
    return [dict(row) for row in rows]


## 2. 指定股票监控测试

这里验证：

- `ORCL` 在监控池里，会被接收；
- `AAPL` 不在监控池里，会被忽略；
- SQLite 里只出现 ORCL。


In [11]:
from news_api.article_fetcher import NoopArticleFetcher
from news_api.config import NewsSettings
from news_api.models import NewsHeadline
from news_api.service import NewsService
from news_api.storage import SQLiteNewsStorage

settings = NewsSettings(db_path=TEST_DB, portfolio_push_score=60, default_push_score=70)
storage = SQLiteNewsStorage(TEST_DB)
bark = RecordingBarkClient()
service = NewsService(
    settings=settings,
    watchlist=TEST_WATCHLIST,
    storage=storage,
    article_fetcher=NoopArticleFetcher(),
    bark_client=bark,
)

service.start()

orcl_ok = service.ingest_headline(
    NewsHeadline(
        symbol="ORCL",
        provider="DJ-N",
        article_id="basic-monitor-001",
        headline="Oracle raises guidance after quarterly results",
        published_at="2026-06-16T09:30:00Z",
        publisher="Dow Jones",
    )
)

aapl_ok = service.ingest_headline(
    NewsHeadline(
        symbol="AAPL",
        provider="DJ-N",
        article_id="basic-monitor-002",
        headline="Apple announces product update",
        published_at="2026-06-16T09:31:00Z",
        publisher="Dow Jones",
    )
)

service.queue.join()
service.stop()

raw_rows = read_table(TEST_DB, "news_raw")
event_rows = read_table(TEST_DB, "news_events")

assert orcl_ok is True
assert aapl_ok is False
assert len(raw_rows) == 1
assert raw_rows[0]["symbol"] == "ORCL"
assert len(event_rows) == 1
assert event_rows[0]["symbol"] == "ORCL"

print("指定股票监控测试通过")
print("raw_rows =", raw_rows)
print("event_rows =", event_rows)


指定股票监控测试通过
raw_rows = [{'unique_key': 'DJ-N:basic-monitor-001', 'provider': 'DJ-N', 'article_id': 'basic-monitor-001', 'ticker_id': None, 'con_id': None, 'symbol': 'ORCL', 'published_at': '2026-06-16T09:30:00Z', 'received_at': '2026-06-16T07:44:51.427510+00:00', 'headline_raw': '', 'headline': 'Oracle raises guidance after quarterly results', 'publisher': 'Dow Jones', 'extra_data': ''}]
event_rows = [{'event_id': 1, 'unique_key': 'DJ-N:basic-monitor-001', 'symbol': 'ORCL', 'provider': 'DJ-N', 'article_id': 'basic-monitor-001', 'headline': 'Oracle raises guidance after quarterly results', 'event_type': 'EARNINGS', 'event_subtypes': '["quarterly results"]', 'relevance_score': 20, 'event_score': 26, 'novelty_score': 15, 'source_score': 10, 'market_impact_score': 8, 'importance_score': 79, 'sentiment': 0.4, 'summary_zh': '["Oracle raises guidance after quarterly results"]', 'reason_important': '命中本地新闻规则，建议进入重点处理。', 'story_id': 'ORCL_DJ-N_d63f5bbc0a', 'should_push': 1, 'created_at': '2026-0

## 3. 实时新闻到推送测试：模拟 IB tickNews

这里模拟 IB 实时回调传入一条新闻，验证完整链路：

`tickNews 数据 -> 清洗标题 -> 入库 -> 评分 -> Bark 推送 -> 推送日志`

这一步不需要 TWS，但测试的是实时回调入口。


In [12]:
TEST_DB_2 = Path(tempfile.mkdtemp()) / "tick_news_push_test.sqlite"
settings_2 = NewsSettings(db_path=TEST_DB_2, portfolio_push_score=60, default_push_score=70)
storage_2 = SQLiteNewsStorage(TEST_DB_2)
bark_2 = RecordingBarkClient()
service_2 = NewsService(
    settings=settings_2,
    watchlist=TEST_WATCHLIST,
    storage=storage_2,
    article_fetcher=NoopArticleFetcher(),
    bark_client=bark_2,
)

service_2.start()
inserted = service_2.ingest_tick_news(
    symbol="ORCL",
    timestamp="2026-06-16T09:35:00Z",
    provider="DJ-N",
    article_id="tick-push-001",
    headline="{A:1:L:en}Oracle raises guidance after quarterly results -- Dow Jones",
    ticker_id=7000,
    extra_data="{}",
)
service_2.queue.join()
service_2.stop()

raw_rows = read_table(TEST_DB_2, "news_raw")
event_rows = read_table(TEST_DB_2, "news_events")
push_rows = read_table(TEST_DB_2, "news_push_log")

assert inserted is True
assert len(raw_rows) == 1
assert raw_rows[0]["headline"] == "Oracle raises guidance after quarterly results"
assert raw_rows[0]["publisher"] == "Dow Jones"
assert len(event_rows) == 1
assert event_rows[0]["importance_score"] >= 60
assert event_rows[0]["should_push"] == 1
assert len(bark_2.calls) == 1
assert bark_2.calls[0]["symbol"] == "ORCL"
assert len(push_rows) == 1
assert push_rows[0]["push_status"] == "ok"

print("实时新闻到推送测试通过")
print("bark_call =", bark_2.calls[0])
print("event =", event_rows[0])
print("push_log =", push_rows[0])


实时新闻到推送测试通过
bark_call = {'symbol': 'ORCL', 'headline': 'Oracle raises guidance after quarterly results', 'event_type': 'EARNINGS', 'importance_score': 79, 'priority': 0}
event = {'event_id': 1, 'unique_key': 'DJ-N:tick-push-001', 'symbol': 'ORCL', 'provider': 'DJ-N', 'article_id': 'tick-push-001', 'headline': 'Oracle raises guidance after quarterly results', 'event_type': 'EARNINGS', 'event_subtypes': '["quarterly results"]', 'relevance_score': 20, 'event_score': 26, 'novelty_score': 15, 'source_score': 10, 'market_impact_score': 8, 'importance_score': 79, 'sentiment': 0.4, 'summary_zh': '["Oracle raises guidance after quarterly results"]', 'reason_important': '命中本地新闻规则，建议进入重点处理。', 'story_id': 'ORCL_DJ-N_d63f5bbc0a', 'should_push': 1, 'created_at': '2026-06-16T07:44:52.358183+00:00'}
push_log = {'push_id': 1, 'unique_key': 'DJ-N:tick-push-001', 'channel': 'bark', 'pushed_at': '2026-06-16T07:44:52.369250+00:00', 'push_status': 'ok', 'response': 'recorded'}


## 4. IBNewsClient.tickNews 回调映射测试

上一步直接调用 `service.ingest_tick_news`。这一节再多测一层：`IBNewsClient.tickNews()` 是否能按 `ticker_id` 找到指定股票，并把新闻送入服务。


In [13]:
from news_api.ib_client import IBNewsClient

TEST_DB_3 = Path(tempfile.mkdtemp()) / "ib_client_tick_test.sqlite"
settings_3 = NewsSettings(db_path=TEST_DB_3, portfolio_push_score=60, default_push_score=70)
storage_3 = SQLiteNewsStorage(TEST_DB_3)
bark_3 = RecordingBarkClient()
service_3 = NewsService(
    settings=settings_3,
    watchlist=TEST_WATCHLIST,
    storage=storage_3,
    article_fetcher=NoopArticleFetcher(),
    bark_client=bark_3,
)

# 不连接 IB，直接构造对象测 tickNews 回调逻辑。
client = IBNewsClient.__new__(IBNewsClient)
client.service = service_3
client.ticker_id_to_symbol = {7000: "ORCL"}

service_3.start()
client.tickNews(
    tickerId=7000,
    timeStamp=1781530200,
    providerCode="DJ-N",
    articleId="ib-tick-001",
    headline="{A:1:L:en}Oracle raises guidance after earnings -- Dow Jones",
    extraData="{}",
)
client.tickNews(
    tickerId=9999,
    timeStamp=1781530200,
    providerCode="DJ-N",
    articleId="ib-tick-ignored",
    headline="Unknown ticker should be ignored",
    extraData="{}",
)
service_3.queue.join()
service_3.stop()

raw_rows = read_table(TEST_DB_3, "news_raw")
event_rows = read_table(TEST_DB_3, "news_events")

assert len(raw_rows) == 1
assert raw_rows[0]["symbol"] == "ORCL"
assert raw_rows[0]["article_id"] == "ib-tick-001"
assert len(event_rows) == 1
assert len(bark_3.calls) == 1

print("IBNewsClient.tickNews 映射测试通过")
print("raw_rows =", raw_rows)
print("bark_call =", bark_3.calls[0])


IBNewsClient.tickNews 映射测试通过
raw_rows = [{'unique_key': 'DJ-N:ib-tick-001', 'provider': 'DJ-N', 'article_id': 'ib-tick-001', 'ticker_id': 7000, 'con_id': None, 'symbol': 'ORCL', 'published_at': '1781530200', 'received_at': '2026-06-16T07:44:53.514832+00:00', 'headline_raw': '{A:1:L:en}Oracle raises guidance after earnings -- Dow Jones', 'headline': 'Oracle raises guidance after earnings', 'publisher': 'Dow Jones', 'extra_data': '{}'}]
bark_call = {'symbol': 'ORCL', 'headline': 'Oracle raises guidance after earnings', 'event_type': 'EARNINGS', 'importance_score': 79, 'priority': 0}


## 5. 可选：真实 Bark 推送测试

如果你要确认手机能收到 Bark，把下面的 `RUN_REAL_BARK_TEST` 改成 `True`，并确保环境变量 `BARK_KEY` 已设置。


In [7]:
RUN_REAL_BARK_TEST = True

if RUN_REAL_BARK_TEST:
    import json
    from news_api.bark_client import BarkClient
    from news_api.models import NewsAnalysis

    key = os.getenv("BARK_KEY", "")
    assert key, "请先设置环境变量 BARK_KEY"

    analysis = NewsAnalysis(
        symbol="ORCL",
        provider="TEST",
        article_id="real-bark-test",
        headline="这是一条 news_api Bark 测试推送",
        event_type="TEST",
        importance_score=88,
        sentiment=0.0,
        summary_zh=["如果你收到这条消息，说明 Bark 推送链路可用。"],
    )

    client = BarkClient(key=key, timeout=30, retries=2, retry_interval=2.0)
    status, response = client.push(analysis, priority=0)
    print("status =", status)
    print("response =", response)

    assert status == "ok", "如果这里是 TLS handshake timeout，说明 api.day.app 网络连接超时，不是新闻流水线没触发。"
    data = json.loads(response)
    assert data.get("code") == 200, data
else:
    print("跳过真实 Bark 推送。需要时把 RUN_REAL_BARK_TEST 改成 True。")


status = ok
response = {"code":200,"message":"success","timestamp":1781595467}


## 6. 可选：真实 IB 实时新闻订阅测试

这个单元会真实连接 TWS / IB Gateway，并订阅 `TEST_WATCHLIST` 里的股票。

注意：真实实时新闻不是每秒都有。如果等待窗口内没有 ORCL 新闻，不代表订阅失败；你至少可以先检查是否成功发出订阅请求。要更快验证“收到新闻后会推送”，用上面的模拟 `tickNews` 测试。


In [ ]:
RUN_LIVE_IB_TEST = True
WAIT_SECONDS = 180

if RUN_LIVE_IB_TEST:
    from news_api.config import SETTINGS, NewsSettings
    from news_api.ib_client import IBNewsClient
    from news_api.service import NewsService
    from news_api.storage import SQLiteNewsStorage
    from news_api.subscription_manager import SubscriptionManager

    live_db = Path(tempfile.mkdtemp()) / "live_ib_news.sqlite"
    live_settings = NewsSettings(
        host=SETTINGS.host,
        port=SETTINGS.port,
        client_id=SETTINGS.client_id,
        provider_codes=SETTINGS.provider_codes,
        db_path=live_db,
        bark_key=os.getenv("BARK_KEY", ""),
        portfolio_push_score=60,
        default_push_score=70,
    )

    live_storage = SQLiteNewsStorage(live_db)
    live_service = NewsService(
        settings=live_settings,
        watchlist=TEST_WATCHLIST,
        storage=live_storage,
    )
    live_client = IBNewsClient(live_service)
    live_client.start_api(live_settings.host, live_settings.port, live_settings.client_id)

    manager = SubscriptionManager(live_client)
    subscribed = manager.subscribe_watchlist(TEST_WATCHLIST, live_settings.provider_codes)
    print("subscribed =", subscribed)
    assert TEST_SYMBOL in subscribed

    deadline = time.time() + WAIT_SECONDS
    try:
        while time.time() < deadline:
            raw_count = len(read_table(live_db, "news_raw"))
            event_count = len(read_table(live_db, "news_events"))
            push_count = len(read_table(live_db, "news_push_log"))
            print(f"raw={raw_count}, events={event_count}, pushes={push_count}")
            if raw_count > 0:
                break
            time.sleep(10)
    finally:
        live_client.stop_api()

    print("live_db =", live_db)
    print("raw =", read_table(live_db, "news_raw"))
    print("events =", read_table(live_db, "news_events"))
    print("push_log =", read_table(live_db, "news_push_log"))
else:
    print("跳过真实 IB 订阅。需要时把 RUN_LIVE_IB_TEST 改成 True。")
